# Silicon OCP Identification With MSMR

This notebook reconstructs the effective silicon OCP from measured composite-anode and graphite lithiation OCP references using the multi-species multi-reaction (MSMR) framework.

The workflow windows the measured OCP curves over the shared voltage range, normalizes stoichiometry, represents `x(U)` with monotone interpolation, fits a multi-gallery MSMR silicon model, and exports the reconstructed silicon OCP used by the diagnostic workflows.

The silicon-fraction bound is based on a preliminary `M = 1` MSMR fit, which gave `alpha ~= 0.37`; the final fit therefore uses a `+/-0.03` tolerance, corresponding to `ALPHA_BOUNDS = (0.34, 0.40)`.


In [ ]:
import os
from datetime import datetime
from pathlib import Path


def find_repo_root(start=None):
    """Find the repository root from the current working directory or notebook folder."""
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "cell_ocp").is_dir() and (candidate / "code").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find repo root. Run this notebook from inside the cloned repository."
    )


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data" / "cell_ocp"
NOTEBOOK_DIR = REPO_ROOT / "code" / "msmr_si_ocp_identification"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(OUTPUT_DIR / ".cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import least_squares, minimize_scalar
from scipy.special import expit
from scipy.signal import savgol_filter
from sklearn.isotonic import IsotonicRegression

print(f"Repository root: {REPO_ROOT}")
print(f"OCP data folder: {DATA_DIR}")
print(f"Output folder: {OUTPUT_DIR}")


In [ ]:
# ---------------------- Configuration ----------------------
ANODE_CSV = DATA_DIR / "Anode_OCP_Lithiation.csv"
GR_CSV = DATA_DIR / "Graphite_OCP_Lithiation_raw.csv"
SIL_CSV = DATA_DIR / "Silicon_OCP_Lithiation_verbrugge.csv"
# Shared voltage window for both measured OCP curves.
U_LOW  = 0.03
U_HIGH = 1.00

# MSMR model settings.
M = 4
T_K = 298.15

# Relative weights for x(U) and dx/dU residuals.
W_X  = 1
W_DX = 0.001

# Parameter bounds. The alpha bounds follow the preliminary M=1 MSMR estimate
# described above.
ALPHA_BOUNDS = (0.34, 0.40)
DELTA_BOUNDS = (-0.002, 0.002)

# Bounds for MSMR gallery sharpness parameters.
OMEGA_MIN = 0.05  
OMEGA_MAX = 4.0
OMEGA0    = 0.06

# Numerical grids for fitting and plotting.
N_FIT_U  = 900
N_PLOT_X = 800

# Savitzky-Golay settings for derivative plots.
SG_WIN  = 41
SG_POLY = 4

# Least-squares convergence settings.
MAX_NFEV = 20000
FTOL = 1e-15
XTOL = 1e-15
GTOL = 1e-15
DIFF_STEP = 1e-6
# -----------------------------------------------------------

R = 8.314
F = 96485

# Optional local residual weighting.
USE_WEIGHT = True

# Domain used to define the locally emphasized residual region.
WEIGHT_DOMAIN = "U"

# Residual emphasis windows in normalized stoichiometry or voltage.
X_FOCUS = (0.05, 1)
U_FOCUS = (0, 0.6)

# Weight multiplier inside the emphasized region.
W_BOOST = 20.0

# Smooth transition width for the weighting window.
W_SMOOTH = 0.01


# ---------------------- Helper functions ----------------------


def save_msmr_params_csv(
    out_dir,
    *,
    alpha_opt,
    delta_opt,
    si_opt,
    M,
    X_opt,
    U_low,
    U_high,
    x_lo,
    x_hi,
    den,
    extra_meta=None,
):
    """
    Save MSMR fit parameters to a timestamped CSV (single row).
    - si_opt[:M]      -> U0
    - exp(si_opt[M:]) -> omega
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = out_dir / f"MSMR_params_{ts}.csv"

    U0 = np.asarray(si_opt[:M], dtype=float)
    omega = np.exp(np.asarray(si_opt[M:2*M], dtype=float))
    X_opt = np.asarray(X_opt, dtype=float)

    row = {
        "timestamp_local": datetime.now().isoformat(timespec="seconds"),
        "M": int(M),
        "alpha": float(alpha_opt),
        "delta": float(delta_opt),
        "U_low": float(U_low),
        "U_high": float(U_high),
        "x_si_U_low": float(x_lo),
        "x_si_U_high": float(x_hi),
        "den": float(den),
        "X_sum": float(X_opt.sum()),
        "U0": np.array2string(U0, separator=",", max_line_width=10_000),
        "omega": np.array2string(omega, separator=",", max_line_width=10_000),
        "X_opt": np.array2string(X_opt, separator=",", max_line_width=10_000),
    }

    if extra_meta:
        row.update(extra_meta)

    df = pd.DataFrame([row])
    df.to_csv(fname, index=False)
    print(f"[saved] {fname}")
    return fname

def load_csv(path: str):
    df = pd.read_csv(path)
    if not {"sto", "p"}.issubset(df.columns):
        raise ValueError(f"{path} must have columns: sto, p")
    x = df["sto"].to_numpy(dtype=float)
    U = df["p"].to_numpy(dtype=float)
    m = np.isfinite(x) & np.isfinite(U)
    return x[m], U[m]


def window_and_normalize_sto(x, U, U_low, U_high):
    """Keep points in [U_low,U_high], then normalize sto to 0-1 inside that subset."""
    m = (U >= U_low) & (U <= U_high)
    xw = x[m].copy()
    Uw = U[m].copy()
    if len(xw) < 20:
        raise ValueError("Too few points in chosen voltage window. Adjust U_LOW/U_HIGH.")
    xw = (xw - np.min(xw)) / (np.max(xw) - np.min(xw) + 1e-12)
    return xw, Uw


class MonotoneXofU:
    """
    Build monotone x(U) using isotonic regression.
    For lithiation, typically x increases as U decreases => x(U) is decreasing in U.
    """
    def __init__(self, U, x, decreasing=True, n=6000):
        idx = np.argsort(U)
        U_s = np.asarray(U)[idx]
        x_s = np.asarray(x)[idx]

        # Downsample dense curves by voltage quantiles before isotonic fitting.
        if len(U_s) > n:
            q = np.linspace(0, 1, n)
            U_q = np.quantile(U_s, q)
            x_q = np.empty_like(U_q)
            for i, u in enumerate(U_q):
                j = np.searchsorted(U_s, u)
                lo = max(0, j - 30)
                hi = min(len(U_s), j + 30)
                x_q[i] = np.median(x_s[lo:hi])
            U_s, x_s = U_q, x_q

        self.Umin = float(np.min(U_s))
        self.Umax = float(np.max(U_s))

        ir = IsotonicRegression(increasing=not decreasing, out_of_bounds="clip")
        ir.fit(U_s, x_s)
        self._predict = ir.predict

    def predict(self, Uq):
        return self._predict(np.asarray(Uq, dtype=float))


def softmax(a):
    a = np.asarray(a, dtype=float)
    a = a - np.max(a)
    ea = np.exp(a)
    return ea / (np.sum(ea) + 1e-12)


def msmr_components_freeX(U, si_params, M, T=T_K):
    """
    si_params = [U0( M ), logw( M ), a_logits( M )]
    X = softmax(a_logits), sum X = 1
    """
    U = np.asarray(U, dtype=float)
    U0 = si_params[:M]
    omega = np.exp(si_params[M:2*M])
    a = si_params[2*M:3*M]
    X = softmax(a)

    k = F / (R * T)
    z = k * (U.reshape(-1, 1) - U0.reshape(1, -1)) / omega.reshape(1, -1)
    s = expit(-z)
    comps = s * X.reshape(1, -1)
    x_si = comps.sum(axis=1)
    return comps, x_si, X


def x_si_msmr_freeX(U, si_params, M, T=T_K):
    _, x_si, _ = msmr_components_freeX(U, si_params, M, T=T_K)
    return x_si


def dxsi_dU_msmr(U, si_params, M, T=T_K):
    """Analytic derivative dx_Si/dU for MSMR sum of sigmoids."""
    U = np.asarray(U, dtype=float)
    U0 = si_params[:M]
    omega = np.exp(si_params[M:2*M])
    a = si_params[2*M:3*M]
    X = softmax(a)

    k = F / (R * T)
    z = k * (U.reshape(-1, 1) - U0.reshape(1, -1)) / omega.reshape(1, -1)
    s = expit(-z)
    ds_dU = -(k / omega.reshape(1, -1)) * s * (1 - s)
    dx_dU = (ds_dU * X.reshape(1, -1)).sum(axis=1)
    return dx_dU


def window_normalize_silicon(U, si_params, M, U_low, U_high):
    """
    Return x_si_raw(U), x_si_tilde(U), den, x_hi, x_lo
    x_si_tilde(U) maps [U_high -> 0, U_low -> 1] within the defined window.
    """
    x_si = x_si_msmr_freeX(U, si_params, M)

    x_hi = x_si_msmr_freeX(np.array([U_high]), si_params, M)[0]
    x_lo = x_si_msmr_freeX(np.array([U_low]),  si_params, M)[0]

    den = (x_lo - x_hi)
    if abs(den) < 1e-6:
        den = np.sign(den) * 1e-6 if den != 0 else 1e-6

    x_tilde = (x_si - x_hi) / den
    x_tilde = np.clip(x_tilde, 0.0, 1.0)
    return x_si, x_tilde, den, x_hi, x_lo


def invert_monotone(x, U, x_grid):
    """Build U(x) by sorting (x,U) pairs then linear interpolate."""
    x = np.asarray(x)
    U = np.asarray(U)
    idx = np.argsort(x)
    x_s = x[idx]
    U_s = U[idx]
    x_u, ui = np.unique(x_s, return_index=True)
    U_u = U_s[ui]
    from scipy.interpolate import interp1d
    f = interp1d(x_u, U_u, kind="linear", bounds_error=False, fill_value=(U_u[0], U_u[-1]))
    return f(x_grid)


def smooth_box(z, z0, z1, width):
    """
    Smooth weighting window with tanh transitions at z0 and z1.
    """
    z = np.asarray(z, float)
    width = max(float(width), 1e-6)
    left  = 0.5 * (1.0 + np.tanh((z - z0) / width))
    right = 0.5 * (1.0 + np.tanh((z1 - z) / width))
    return left * right


def build_weights(U_grid, x_meas, use_weight=True):
    """
    Return per-point weights w_x(U) and w_dx(U) for residuals.
    """
    if not use_weight:
        w = np.ones_like(U_grid, dtype=float)
        return w, w

    if WEIGHT_DOMAIN.lower() == "x":
        z = x_meas
        z0, z1 = X_FOCUS
        width = W_SMOOTH
    else:
        z = U_grid
        z0, z1 = U_FOCUS
        width = W_SMOOTH

    box = smooth_box(z, z0, z1, width)
    w = 1.0 + (W_BOOST - 1.0) * box

    w_x = w
    w_dx = w

    return w_x, w_dx


# Joint residual used for least-squares fitting.
def residual_joint(p, U_grid, x_an_of_U, x_gr_of_U, M, U_low, U_high, w_x, w_dx):
    """
    Joint residual:
      r = [ w_x_global * w_x_local * (x_model-x_meas) ;
            w_dx_global * w_dx_local * (dx_model/dU - dx_meas/dU) ]
    where x_model uses window-normalized silicon x_si_tilde(U).
    """
    si_params = p[:3*M]
    alpha = p[3*M]
    delta = p[3*M + 1]

    x_meas = x_an_of_U.predict(U_grid)
    x_gr   = x_gr_of_U.predict(U_grid)

    # Use window-normalized silicon stoichiometry in the composite anode model.
    x_si_raw, x_si_tilde, den, _, _ = window_normalize_silicon(U_grid, si_params, M, U_low, U_high)

    x_model = delta + (1 - alpha) * x_gr + alpha * x_si_tilde
    x_model = np.clip(x_model, 0.0, 1.0)

    # Derivative residual in x(U) space.
    dxmeas_dU = np.gradient(x_meas, U_grid)
    dxgr_dU   = np.gradient(x_gr,   U_grid)

    dxsi_dU_raw = dxsi_dU_msmr(U_grid, si_params, M)
    dxsi_dU_tilde = dxsi_dU_raw / den

    dxmodel_dU = (1 - alpha) * dxgr_dU + alpha * dxsi_dU_tilde

    # Pointwise local weighting.
    wloc_x, wloc_dx = build_weights(U_grid, x_meas, use_weight=USE_WEIGHT)

    r_x  = (w_x  * wloc_x)  * (x_model - x_meas)
    r_dx = (w_dx * wloc_dx) * (dxmodel_dU - dxmeas_dU)

    return np.r_[r_x, r_dx]

def savgol_derivative(x, y, window, polyorder):
    from scipy.signal import savgol_filter
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    dydx = np.gradient(y, x)
    dydx = savgol_filter(dydx, window_length=window, polyorder=polyorder)
    return dydx

# ---------------------- Main analysis ----------------------
def main():
    # Load measured OCP curves.
    x_an_raw, U_an_raw = load_csv(ANODE_CSV)
    x_gr_raw, U_gr_raw = load_csv(GR_CSV)

    # Use the same voltage window and stoichiometry normalization for both curves.
    x_an, U_an = window_and_normalize_sto(x_an_raw, U_an_raw, U_LOW, U_HIGH)
    x_gr, U_gr = window_and_normalize_sto(x_gr_raw, U_gr_raw, U_LOW, U_HIGH)

    print(f"Anode window:    N={len(U_an)}  U=[{U_an.min():.4f},{U_an.max():.4f}]")
    print(f"Graphite window: N={len(U_gr)}  U=[{U_gr.min():.4f},{U_gr.max():.4f}]")

    # Build monotone x(U) representations of the measured curves.
    x_an_of_U = MonotoneXofU(U_an, x_an, decreasing=True, n=6000)
    x_gr_of_U = MonotoneXofU(U_gr, x_gr, decreasing=True, n=6000)

    # Fit only over the common voltage support of the anode and graphite curves.
    Umin = max(x_an_of_U.Umin, x_gr_of_U.Umin, U_LOW)
    Umax = min(x_an_of_U.Umax, x_gr_of_U.Umax, U_HIGH)
    U_grid = np.linspace(Umin, Umax, N_FIT_U)

    # Assemble optimizer bounds.
    lb = np.r_[np.ones(M) * Umin,
               np.ones(M) * np.log(OMEGA_MIN),
               np.ones(M) * (-10.0),
               ALPHA_BOUNDS[0],
               DELTA_BOUNDS[0]]

    ub = np.r_[np.ones(M) * Umax,
               np.ones(M) * np.log(OMEGA_MAX),
               np.ones(M) * (10.0),
               ALPHA_BOUNDS[1],
               DELTA_BOUNDS[1]]

    # Feasible initial parameter vector.
    U0_init = np.linspace(Umin + 0.02, Umax - 0.02, M)
    omega0 = min(max(OMEGA0, OMEGA_MIN * 1.5), OMEGA_MAX / 2)
    logw_init = np.log(np.ones(M) * omega0)
    a_init = np.zeros(M)
    alpha0 = np.clip(0.45, ALPHA_BOUNDS[0] + 1e-6, ALPHA_BOUNDS[1] - 1e-6)
    delta0 = np.clip(0.0,  DELTA_BOUNDS[0] + 1e-6, DELTA_BOUNDS[1] - 1e-6)

    p0 = np.r_[U0_init, logw_init, a_init, alpha0, delta0]
    p0 = np.minimum(np.maximum(p0, lb + 1e-9), ub - 1e-9)

    # Jointly fit x(U) and dx/dU.
    res = least_squares(
        residual_joint,
        p0,
        bounds=(lb, ub),
        args=(U_grid, x_an_of_U, x_gr_of_U, M, U_LOW, U_HIGH, W_X, W_DX),
        method="trf",
        ftol=FTOL, xtol=XTOL, gtol=GTOL,
        x_scale="jac",
        diff_step=DIFF_STEP,
        max_nfev=MAX_NFEV,
        verbose=2
    )

    p_opt = res.x
    si_opt = p_opt[:3*M]
    alpha_opt = p_opt[3*M]
    delta_opt = p_opt[3*M + 1]

    # Evaluate fitted model components and reconstructed anode curve.
    comps, x_si_raw_U, X_opt = msmr_components_freeX(U_grid, si_opt, M)
    x_si_raw_U, x_si_tilde_U, den, x_hi, x_lo = window_normalize_silicon(U_grid, si_opt, M, U_LOW, U_HIGH)

    x_an_meas_U = x_an_of_U.predict(U_grid)
    x_gr_U = x_gr_of_U.predict(U_grid)

    x_an_model_U = delta_opt + (1 - alpha_opt) * x_gr_U + alpha_opt * x_si_tilde_U
    x_an_model_U = np.clip(x_an_model_U, 0.0, 1.0)

    print("\n===== FIT RESULTS =====")
    print("alpha =", alpha_opt)
    print("delta =", delta_opt)
    print("U0    =", si_opt[:M])
    print("omega =", np.exp(si_opt[M:2*M]))
    print("X_opt =", X_opt, "sum=", X_opt.sum())
    print(f"Si window endpoints: x_si(U_high)={x_hi:.4f}, x_si(U_low)={x_lo:.4f}, den={den:.4f}")
    
    save_msmr_params_csv(
    out_dir=OUTPUT_DIR / "MSMR_saved_params",
    alpha_opt=alpha_opt,
    delta_opt=delta_opt,
    si_opt=si_opt,
    M=M,
    X_opt=X_opt,
    U_low=U_LOW,
    U_high=U_HIGH,
    x_lo=x_lo,
    x_hi=x_hi,
    den=den,
    extra_meta={
        "note": "MSMR-Si OCP identification"
    },
)

    # Plot fitted anode and silicon OCP results.
    # Reconstruct U(x) in window-normalized coordinates.
    x_plot = np.linspace(0, 1, N_PLOT_X)
    U_an_meas  = invert_monotone(x_an_meas_U,  U_grid, x_plot)
    U_an_model = invert_monotone(x_an_model_U, U_grid, x_plot)

    plt.figure()
    plt.plot(x_plot, U_an_meas, label="Measured anode OCP")
    plt.plot(x_plot, U_an_model, label="Reconstructed anode OCP")
    plt.xlabel("x_an (normalized in window)")
    plt.ylabel("U_an (V vs Li/Li+)")
    plt.title("U(x) comparison (window-normalized)")
    plt.legend()
    
    abs_err_U = np.abs(U_an_model - U_an_meas)
    rmse_fig1 = np.sqrt(np.mean((U_an_model - U_an_meas)**2))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].plot(x_plot, U_an_meas, label="Measured anode OCP")
    axes[0].plot(x_plot, U_an_model, label="Reconstructed anode OCP")
    axes[0].set_xlabel("x_an (normalized in window)")
    axes[0].set_ylabel("U_an (V vs Li/Li+)")
    axes[0].set_title("U(x) comparison")
    axes[0].legend()

    axes[1].plot(x_plot, abs_err_U, label="Absolute error")
    axes[1].set_xlabel("x_an (normalized in window)")
    axes[1].set_ylabel("|U_model - U_meas| (V)")
    axes[1].set_title(f"Absolute error (RMSE = {rmse_fig1:.3e} V)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    # Estimated silicon OCP using window-normalized silicon stoichiometry.
    U_si = invert_monotone(x_si_tilde_U, U_grid, x_plot)
    plt.figure()
    plt.plot(x_plot, U_si)
    plt.xlabel("x_Si (normalized in window)")
    plt.ylabel("U_Si (V vs Li/Li+)")
    plt.title("Estimated silicon OCP (MSMR, window-normalized sto)")

    # Differential OCP comparison.
    dx = x_plot[1] - x_plot[0]
    win = SG_WIN
    if win >= len(x_plot):
        win = len(x_plot) - 1 if (len(x_plot) % 2 == 0) else len(x_plot)
    if win % 2 == 0:
        win += 1

    dVdx_meas  = savgol_filter(U_an_meas,  window_length=win, polyorder=SG_POLY, deriv=1, delta=dx, mode="interp")
    dVdx_model = savgol_filter(U_an_model, window_length=win, polyorder=SG_POLY, deriv=1, delta=dx, mode="interp")

    plt.figure()
    plt.plot(x_plot, -dVdx_meas, label="Measured dV/dx")
    plt.plot(x_plot, -dVdx_model, label="Model dV/dx")
    plt.ylim([0,1])
    plt.xlabel("x_an (normalized in window)")
    plt.ylabel("dV/dx (V per x)")
    plt.title(f"dV/dx comparison (SavGol win={win}, poly={SG_POLY})")
    plt.legend()

    # MSMR gallery contributions before window normalization.
    plt.figure()
    for j in range(M):
        plt.plot(U_grid, comps[:, j], label=f"comp {j+1} (X={X_opt[j]:.3f})")
    plt.plot(U_grid, x_si_raw_U, linewidth=2.5, label="Si total (raw)")
    plt.gca().invert_xaxis()
    plt.xlabel("U (V vs Li/Li+)")
    plt.ylabel("x contribution")
    plt.title("MSMR gallery contributions (raw Si x(U))")
    plt.legend(ncol=2)

    plt.show()

    # Save tabulated outputs for plotting and reuse.
    out = pd.DataFrame({
        "U": U_grid,
        "x_an_meas": x_an_meas_U,
        "x_an_model": x_an_model_U,
        "x_gr": x_gr_U,
        "x_si_raw": x_si_raw_U,
        "x_si_tilde": x_si_tilde_U,
    })
    out.to_csv(OUTPUT_DIR / "fit_diagnostics_x_of_U.csv", index=False)

    comp_df = pd.DataFrame({"U": U_grid})
    for j in range(M):
        comp_df[f"x_si_comp_{j+1}"] = comps[:, j]
    comp_df["x_si_total_raw"] = x_si_raw_U
    comp_df.to_csv(OUTPUT_DIR / "fit_diagnostics_si_components.csv", index=False)

    print(f"\nSaved intermediate fit diagnostics to: {OUTPUT_DIR}")

    # Figure 1: measured vs reconstructed anode OCP, U(x)
    fig1_df = pd.DataFrame({
        "x_plot": x_plot,
        "U_an_meas": U_an_meas,
        "U_an_model": U_an_model,
    })
    fig1_df.to_csv(OUTPUT_DIR / "fig1_anode_OCP_Ux.csv", index=False)

    # Figure 2: estimated silicon OCP, U_si(x)
    fig2_df = pd.DataFrame({
        "x_plot": x_plot,
        "U_si": U_si,
    })
    fig2_df.to_csv(OUTPUT_DIR / "fig2_silicon_OCP_Ux.csv", index=False)

    # Figure 3: dV/dx comparison
    fig3_df = pd.DataFrame({
        "x_plot": x_plot,
        "minus_dVdx_meas": -dVdx_meas,
        "minus_dVdx_model": -dVdx_model,
    })
    fig3_df.to_csv(OUTPUT_DIR / "fig3_dVdx_comparison.csv", index=False)

    # Figure 4: MSMR components
    fig4_df = pd.DataFrame({"U": U_grid})
    for j in range(M):
        fig4_df[f"x_si_comp_{j+1}"] = comps[:, j]
    fig4_df["x_si_total_raw"] = x_si_raw_U
    fig4_df.to_csv(OUTPUT_DIR / "fig4_MSMR_components.csv", index=False)

    # RMSE metrics for the reconstructed anode OCP.
    rmse_fig1 = np.sqrt(np.mean((U_an_model - U_an_meas)**2))
    mask_fig1_lowU = U_an_meas <= 0.4
    rmse_fig1_lowU = np.sqrt(np.mean((U_an_model[mask_fig1_lowU] - U_an_meas[mask_fig1_lowU])**2))

    print("\n===== RMSE =====")
    print(f"Figure 1 RMSE [U(x), full]      = {rmse_fig1:.6e} V")
    print(f"Figure 1 RMSE [U(x), U<=0.4V]   = {rmse_fig1_lowU:.6e} V")
    print(f"Figure 1 points with U<=0.4V: {np.sum(mask_fig1_lowU)} / {len(U_an_meas)}")

    # Literature-Si baseline: fit alpha only in x(U) space with delta fixed to 0.
    # Build literature Si x(U) from the reference silicon OCP curve.
    x_si_lit_raw, U_si_lit_raw = load_csv(SIL_CSV)
    x_si_lit, U_si_lit = window_and_normalize_sto(x_si_lit_raw, U_si_lit_raw, U_LOW, U_HIGH)
    U_si_lit_x = invert_monotone(x_si_lit, U_si_lit, x_plot)

    x_si_lit_of_U = MonotoneXofU(
        U_si_lit_x,
        x_plot,
        decreasing=True
    )

    x_si_lit_U = x_si_lit_of_U.predict(U_grid)
    x_si_lit_U = np.clip(x_si_lit_U, 0.0, 1.0)

    # Fit baseline alpha by minimizing the final U(x) RMSE.
    def objective_alpha_only(alpha):
        x_an_model_lit_U = (1 - alpha) * x_gr_U + alpha * x_si_lit_U
        U_an_model_lit = invert_monotone(x_an_model_lit_U, U_grid, x_plot)
        rmse = np.sqrt(np.mean((U_an_model_lit - U_an_meas)**2))
        return rmse

    # Alpha is refit for the literature-Si baseline with a broader interval.
    alpha_lb, alpha_ub = 0.3, 0.51

    res_alpha_lit = minimize_scalar(
        objective_alpha_only,
        bounds=(alpha_lb, alpha_ub),
        method="bounded",
        options={"xatol": 1e-4}
    )

    alpha_lit_opt = res_alpha_lit.x

    print("\n===== Literature-Si baseline fit (delta = 0) =====")
    print(f"alpha_lit_opt = {alpha_lit_opt:.6f}")

    # Reconstruct the literature-Si baseline using the fitted alpha.
    x_an_model_lit_U = (1 - alpha_lit_opt) * x_gr_U + alpha_lit_opt * x_si_lit_U
    U_an_model_lit = invert_monotone(x_an_model_lit_U, U_grid, x_plot)

    # Compute differential OCP curves on a consistent x grid.
    dVdx_model_lit = np.gradient(U_an_model_lit, x_plot)
    dVdx_model_lit = savgol_filter(dVdx_model_lit, window_length=SG_WIN, polyorder=SG_POLY)

    dVdx_meas_cmp = np.gradient(U_an_meas, x_plot)
    dVdx_meas_cmp = savgol_filter(dVdx_meas_cmp, window_length=SG_WIN, polyorder=SG_POLY)

    dVdx_model_cmp = np.gradient(U_an_model, x_plot)
    dVdx_model_cmp = savgol_filter(dVdx_model_cmp, window_length=SG_WIN, polyorder=SG_POLY)

    # Compare baseline and MSMR reconstructions.
    rmse_anode_lit = np.sqrt(np.mean((U_an_model_lit - U_an_meas)**2))
    rmse_anode_msmr = np.sqrt(np.mean((U_an_model - U_an_meas)**2))
    print(len(dVdx_model_cmp))
    rmse_dvdx_lit = np.sqrt(np.mean((dVdx_model_lit[20:-20] - dVdx_meas_cmp[20:-20] )**2))
    rmse_dvdx_msmr = np.sqrt(np.mean((dVdx_model_cmp[20:-20]  - dVdx_meas_cmp[20:-20] )**2))

    print("\n===== Baseline vs MSMR =====")
    print(f"Anode OCP RMSE (literature Si) = {rmse_anode_lit:.6e} V")
    print(f"Anode OCP RMSE (MSMR Si)       = {rmse_anode_msmr:.6e} V")
    print(f"dV/dx RMSE (literature Si)     = {rmse_dvdx_lit:.6e} V")
    print(f"dV/dx RMSE (MSMR Si)           = {rmse_dvdx_msmr:.6e} V")
    print(f"graphite fraction (literature) = {1 - alpha_lit_opt:.6f}")
    print(f"graphite fraction (MSMR)       = {1 - alpha_opt:.6f}")

    # Save baseline summary metrics.
    cmp_df = pd.DataFrame({
        "case": ["literature_Si", "MSMR_Si"],
        "alpha_silicon": [alpha_lit_opt, alpha_opt],
        "graphite_fraction": [1 - alpha_lit_opt, 1 - alpha_opt],
        "delta": [0.0, delta_opt],
        "RMSE_anode_OCP_V": [rmse_anode_lit, rmse_anode_msmr],
        "RMSE_dVdx_V": [rmse_dvdx_lit, rmse_dvdx_msmr]
    })
    cmp_df.to_csv(OUTPUT_DIR / "baseline_vs_msmr_comparison.csv", index=False)

    # Plot anode OCP comparison.
    plt.figure(figsize=(6.8, 4.6))
    plt.plot(x_plot, U_an_meas, label="Measured anode OCP", linewidth=2.2)
    plt.plot(x_plot, U_an_model_lit,
            label=f"Literature Si + graphite (RMSE={rmse_anode_lit*1e3:.1f} mV)",
            linewidth=2.0)
    plt.plot(x_plot, U_an_model,
            label=f"MSMR Si + graphite (RMSE={rmse_anode_msmr*1e3:.1f} mV)",
            linewidth=2.0)
    plt.xlabel("x_an (normalized in window)")
    plt.ylabel("U_an (V vs Li/Li+)")
    plt.title("Anode OCP reconstruction: literature Si vs MSMR Si")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Plot differential OCP comparison.
    plt.figure(figsize=(6.8, 4.6))
    plt.plot(x_plot, -dVdx_meas_cmp, label="Measured", linewidth=2.2)
    plt.plot(x_plot, -dVdx_model_lit,
            label=f"Literature Si + graphite (RMSE={rmse_dvdx_lit*1e3:.1f} mV)",
            linewidth=2.0)
    plt.plot(x_plot, -dVdx_model_cmp,
            label=f"MSMR Si + graphite (RMSE={rmse_dvdx_msmr*1e3:.1f} mV)",
            linewidth=2.0)
    plt.xlabel("x_an (normalized in window)")
    plt.ylabel("-dV/dx")
    plt.title("Differential anode OCP: literature Si vs MSMR Si")
    plt.legend()
    plt.tight_layout()
    plt.ylim([0,1])
    plt.show()

    # Save baseline comparison data.
    baseline_plot_df = pd.DataFrame({
        "x_plot": x_plot,
        "U_an_meas": U_an_meas,
        "U_an_model_lit": U_an_model_lit,
        "U_an_model_msmr": U_an_model,
        "minus_dVdx_meas": -dVdx_meas_cmp,
        "minus_dVdx_lit": -dVdx_model_lit,
        "minus_dVdx_msmr": -dVdx_model_cmp
    })
    baseline_plot_df.to_csv(OUTPUT_DIR / "baseline_vs_msmr_plotdata.csv", index=False)

    print(f"Saved baseline comparison files to: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()
    
    

In [ ]:
import numpy as np
import pandas as pd

FIT_X_CSV = OUTPUT_DIR / "fit_diagnostics_x_of_U.csv"
GR_TARGET_CSV = DATA_DIR / "Graphite_OCP_Lithiation.csv"
OUT_CSV = OUTPUT_DIR / "Silicon_OCP_Lithiation_MSMR.csv"

U_LOW  = 0.03
U_HIGH = 1.00

# Load fitted silicon x(U).
fit = pd.read_csv(FIT_X_CSV)
U_fit = fit["U"].to_numpy(float)

if "x_si_tilde" in fit.columns:
    x_si_fit = fit["x_si_tilde"].to_numpy(float)
    si_col = "x_si_tilde"
elif "x_si_raw" in fit.columns:
    x_si_fit = fit["x_si_raw"].to_numpy(float)
    si_col = "x_si_raw"
elif "x_si" in fit.columns:
    x_si_fit = fit["x_si"].to_numpy(float)
    si_col = "x_si"
else:
    raise ValueError("fit_diagnostics_x_of_U.csv must contain x_si_tilde or x_si_raw/x_si")

# Sort by voltage for interpolation.
idx = np.argsort(U_fit)
U_fit = U_fit[idx]
x_si_fit = x_si_fit[idx]

# Use the graphite voltage grid as the export target.
gr = pd.read_csv(GR_TARGET_CSV)
U_tar = gr["p"].to_numpy(float)
mask = np.isfinite(U_tar) & (U_tar >= U_LOW) & (U_tar <= U_HIGH)
U_tar = U_tar[mask]

# Interpolate without extrapolation by clamping to the fitted voltage range.
U_tar_clamped = np.clip(U_tar, U_fit.min(), U_fit.max())
x_si_on_tar = np.interp(U_tar_clamped, U_fit, x_si_fit)

out = pd.DataFrame({"sto": x_si_on_tar, "p": U_tar})
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (Si from {si_col}, linear interp)")
